# 📊 Module 1.5: Image Histograms — Statistical View of Images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_01_Image_Foundations/05_Image_Histograms/image_histograms.ipynb)

---

## 🎯 Learning Objectives
1. Compute and interpret image histograms
2. Histogram equalization with full math derivation
3. Histogram as RL state feature
4. Histogram matching as an RL target

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless -q

import numpy as np
import matplotlib.pyplot as plt

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# Real images from scikit-image (built-in, no download needed)
astronaut_img = data.astronaut()       # 512x512x3 real photo
camera_img = data.camera()             # 512x512 grayscale real photo  
coins_img = data.coins()               # Real coins photograph
coffee_img = data.coffee()             # 400x600x3 real photo

# MNIST - 70,000 real handwritten digits (auto-downloads ~11MB)
mnist_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST loaded: {len(mnist_dataset)} real handwritten digit images (28x28)")
print(f"✅ scikit-image loaded: astronaut {astronaut_img.shape}, camera {camera_img.shape}")

## Deep Mathematical Derivation: Histogram Equalization

### Step 1: Probability Density of Pixel Intensities
For an image with $N$ total pixels and $n_k$ pixels with intensity $k$:
$$p(k) = \frac{n_k}{N}, \quad k = 0, 1, \ldots, L-1$$

where $L = 256$ for 8-bit images. This is a valid PMF since $\sum_{k=0}^{L-1} p(k) = 1$.

### Step 2: Cumulative Distribution Function (CDF)
$$c(k) = \sum_{j=0}^{k} p(j) = \sum_{j=0}^{k} \frac{n_j}{N}$$

**Properties:** $c(0) \geq 0$, $c(L-1) = 1$, and $c$ is monotonically non-decreasing.

### Step 3: Histogram Equalization Transform (Full Derivation)

**Goal:** Find a transformation $T(k)$ such that the output image has a uniform histogram.

**Step 3a:** Model intensities as continuous r.v. $r \in [0, 1]$ with PDF $p_r(r)$. We want to find $s = T(r)$ such that $p_s(s) = 1$ (uniform).

**Step 3b:** Use the change-of-variables formula:
$$p_s(s) = p_r(r) \left|\frac{dr}{ds}\right|$$

**Step 3c:** For $p_s(s) = 1$, we need:
$$\frac{ds}{dr} = p_r(r)$$

**Step 3d:** Integrate both sides:
$$s = T(r) = \int_0^r p_r(w) \, dw = \text{CDF}(r) \quad \blacksquare$$

**Step 3e:** Discrete version:
$$T(k) = \text{round}\left((L-1) \cdot c(k)\right) = \text{round}\left((L-1) \sum_{j=0}^{k} \frac{n_j}{N}\right)$$

### Step 4: Proof That This Achieves Uniformity

**Claim:** If $s = \text{CDF}(r)$ and $r \sim p_r$, then $s \sim \text{Uniform}[0,1]$.

**Proof:**
$$P(s \leq s_0) = P(\text{CDF}(r) \leq s_0) = P(r \leq \text{CDF}^{-1}(s_0)) = \text{CDF}(\text{CDF}^{-1}(s_0)) = s_0$$

Therefore $s$ has CDF $F_s(s_0) = s_0$, which is the uniform distribution. $\blacksquare$

### Step 5: Histogram Matching (Specification) — Generalization
**Goal:** Transform image $A$ so its histogram matches image $B$'s histogram.

**Step 5a:** Equalize $A$: $s = T_A(r_A) = \text{CDF}_A(r_A)$

**Step 5b:** Compute inverse of $B$'s equalization: $r_B = T_B^{-1}(s) = \text{CDF}_B^{-1}(s)$

**Step 5c:** Composite mapping: $r_B = T_B^{-1}(T_A(r_A)) = \text{CDF}_B^{-1}(\text{CDF}_A(r_A))$

**Proof of correctness:** The output $r_B$ has the same CDF as image $B$, since:
$$P(r_B \leq t) = P(\text{CDF}_B^{-1}(\text{CDF}_A(r_A)) \leq t) = P(\text{CDF}_A(r_A) \leq \text{CDF}_B(t)) = \text{CDF}_B(t) \quad \blacksquare$$

### Step 6: Entropy and Histogram Shape
**Shannon entropy** of the image:
$$H = -\sum_{k=0}^{L-1} p(k) \log_2 p(k)$$

**Theorem:** Entropy is maximized when the histogram is uniform.

**Proof:** By the method of Lagrange multipliers, maximize $H$ subject to $\sum p(k) = 1$:
$$\frac{\partial}{\partial p(k)}\left[-\sum p(k)\log p(k) - \lambda\left(\sum p(k) - 1\right)\right] = -\log p(k) - 1 - \lambda = 0$$
$$\Rightarrow p(k) = e^{-1-\lambda} = \text{const} = \frac{1}{L} \quad \blacksquare$$

### RL Connection: Histogram as Compact State Representation
Instead of using the full image as the RL state ($256^{H \times W}$ states), use the histogram:
$$\mathbf{h} = [p(0), p(1), \ldots, p(255)] \in \mathbb{R}^{256}$$

This reduces a 64×64 grayscale image from $256^{4096}$ states to a 256-dimensional continuous state — a **massive** compression that makes RL feasible!

## Histogram as Probability Distribution — Information-Theoretic Analysis

### Step 1: From Counts to Probability Mass Function

The normalized histogram defines a valid PMF over the intensity alphabet $\mathcal{X} = \{0, 1, \ldots, L-1\}$:

$$p_X(k) = \frac{n_k}{N} = \frac{|\{(x,y) : I(x,y) = k\}|}{H \times W}$$

**Verification of PMF axioms:**
1. Non-negativity: $p_X(k) = n_k / N \geq 0$ since $n_k \geq 0$ $\checkmark$
2. Normalization: $\sum_{k=0}^{L-1} p_X(k) = \frac{1}{N} \sum_{k=0}^{L-1} n_k = \frac{N}{N} = 1$ $\checkmark$

### Step 2: Statistical Moments from the Histogram

The histogram encodes all intensity statistics of the image:

**First moment (mean brightness):**
$$\mu = E[X] = \sum_{k=0}^{L-1} k \cdot p_X(k)$$

**Second central moment (variance = contrast measure):**
$$\sigma^2 = E[(X - \mu)^2] = \sum_{k=0}^{L-1} (k - \mu)^2 \cdot p_X(k)$$

**Third standardized moment (skewness = brightness asymmetry):**
$$\gamma_1 = \frac{E[(X-\mu)^3]}{\sigma^3} = \frac{1}{\sigma^3}\sum_{k=0}^{L-1} (k - \mu)^3 p_X(k)$$

- $\gamma_1 > 0$: right-skewed (mostly dark with bright outliers)
- $\gamma_1 < 0$: left-skewed (mostly bright with dark outliers)
- $\gamma_1 = 0$: symmetric

**Fourth standardized moment (kurtosis = tail weight):**
$$\gamma_2 = \frac{E[(X-\mu)^4]}{\sigma^4} - 3$$

High kurtosis indicates extreme pixel values (high dynamic range content).

### Step 3: Shannon Entropy — Information Content of an Image

$$H(X) = -\sum_{k=0}^{L-1} p_X(k) \log_2 p_X(k)$$

**Interpretation:** $H(X)$ measures the average number of bits needed to encode one pixel.

**Bounds:**
- **Minimum entropy:** $H = 0$ when the image is a single constant value ($p(k_0) = 1$ for some $k_0$)
- **Maximum entropy:** $H = \log_2 L = 8$ bits for 8-bit images, achieved when $p(k) = 1/L$ $\forall k$ (uniform)

**Proof of maximum entropy (Lagrange multipliers):**

Maximize $H = -\sum_k p_k \ln p_k$ subject to $\sum_k p_k = 1$:

$$\mathcal{L} = -\sum_k p_k \ln p_k - \lambda\left(\sum_k p_k - 1\right)$$

$$\frac{\partial \mathcal{L}}{\partial p_k} = -\ln p_k - 1 - \lambda = 0 \implies p_k = e^{-1-\lambda}$$

Since this is constant for all $k$: $p_k = 1/L$ for all $k$. $\blacksquare$

### Step 4: Relationship to Lossless Compression

By Shannon's source coding theorem, the minimum average codeword length for encoding pixel values satisfies:
$$H(X) \leq \bar{L} < H(X) + 1$$

An image with entropy $H = 4$ bits/pixel can be losslessly compressed to roughly half of its 8-bit representation — the histogram tells us exactly how compressible the image is.

## CDF Properties, Monotonicity, and the Probability Integral Transform

The CDF is the mathematical engine behind histogram equalization. Its properties guarantee that equalization works.

### Step 1: CDF Construction and Monotonicity Proof

**Definition:** The CDF of the intensity distribution:
$$F(k) = P(X \leq k) = \sum_{j=0}^{k} p(j) = \sum_{j=0}^{k} \frac{n_j}{N}$$

**Proof of monotonicity:** For $k_1 < k_2$:
$$F(k_2) - F(k_1) = \sum_{j=k_1+1}^{k_2} p(j) \geq 0$$

since $p(j) \geq 0$ for all $j$. Therefore $F$ is monotonically non-decreasing. $\blacksquare$

**Strict monotonicity** holds when every intensity level is present in the image (i.e., $n_j > 0$ for all $j$ in $[k_1+1, k_2]$).

### Step 2: The Probability Integral Transform (Full Proof)

**Theorem:** If $X$ is a continuous random variable with CDF $F_X$, then $Y = F_X(X) \sim \text{Uniform}(0, 1)$.

**Proof:**
$$P(Y \leq y) = P(F_X(X) \leq y) = P(X \leq F_X^{-1}(y)) = F_X(F_X^{-1}(y)) = y$$

for $y \in [0, 1]$. Since $P(Y \leq y) = y$ is the CDF of $\text{Uniform}(0,1)$, we conclude $Y \sim \text{Uniform}(0,1)$. $\blacksquare$

**This is the theoretical foundation of histogram equalization.** The equalization transform $T(k) = \lfloor (L-1) \cdot F(k) \rceil$ is the discrete approximation of the probability integral transform.

### Step 3: Why Discrete Equalization Is Only Approximately Uniform

In the continuous case, the probability integral transform yields an exactly uniform distribution. In the discrete case, perfect uniformity is generally impossible.

**Proof:** Consider an image with $N$ pixels and $L$ intensity levels. After equalization, each output level $j$ should have $N/L$ pixels. But $N/L$ may not be an integer, and the CDF mapping $T(k) = \lfloor (L-1) F(k) \rceil$ can map multiple input levels to the same output level.

**Example:** If $p(0) = 0.4$ and $L = 256$, then $T(0) = \lfloor 255 \times 0.4 \rceil = 102$. All pixels at intensity 0 map to intensity 102 — output level 102 gets 40% of all pixels, far from uniform.

**Bound on non-uniformity:** The maximum deviation from uniform is bounded by the maximum input probability:
$$\max_k |p_{\text{out}}(k) - 1/L| \leq \max_k p_{\text{in}}(k) \quad \blacksquare$$

### Step 4: Histogram Specification (Matching) — Complete Algorithm

**Goal:** Transform image $A$ so its histogram matches a target histogram $h_{\text{target}}$.

**Algorithm:**
1. Compute $F_A(k) = \sum_{j=0}^k p_A(j)$ (source CDF)
2. Compute $F_T(k) = \sum_{j=0}^k p_T(j)$ (target CDF)
3. For each source level $k$: find $T(k) = F_T^{-1}(F_A(k)) = \arg\min_m |F_T(m) - F_A(k)|$
4. Map: $I_{\text{out}}(x,y) = T(I_{\text{in}}(x,y))$

**Proof of correctness:** 
$$F_{\text{out}}(k) = P(T(X) \leq k) = P(X \leq T^{-1}(k)) = F_A(F_A^{-1}(F_T(k))) = F_T(k) \quad \blacksquare$$

The output has the same CDF (and hence histogram) as the target. This generalizes equalization (where the target is uniform).

## 1. Image Histogram

### Definition:
The histogram $h(k)$ of a digital image counts the number of pixels with intensity $k$:

$$h(k) = |\{(x, y) : I(x, y) = k\}|, \quad k = 0, 1, ..., L-1$$

### Normalized histogram (probability):
$$p(k) = \frac{h(k)}{M \times N}$$

Where $M \times N$ is total pixels. This gives us a **probability distribution** of intensities.

### Statistics from histogram:
- **Mean:** $\mu = \sum_{k=0}^{L-1} k \cdot p(k)$
- **Variance:** $\sigma^2 = \sum_{k=0}^{L-1} (k - \mu)^2 \cdot p(k)$
- **Entropy:** $H = -\sum_{k=0}^{L-1} p(k) \log_2 p(k)$

## 2. Histogram Equalization

### Goal: Transform image to have uniform histogram

**CDF (Cumulative Distribution Function):**
$$c(k) = \sum_{j=0}^{k} p(j)$$

**Equalization transform:**
$$T(k) = \text{round}\left((L-1) \cdot c(k)\right)$$

**New image:**
$$I_{eq}(x, y) = T(I(x, y))$$

### For RL:
Histogram equalization is one **action** an RL agent might learn to apply. The histogram itself serves as a compact **state descriptor** (256 values instead of millions of pixels!).

## Histogram Equalization of Color Images — Channel Independence and Alternatives

Applying histogram equalization to color images is more complex than grayscale. Naive approaches can distort colors.

### Step 1: The Problem with Per-Channel Equalization

Equalizing R, G, B channels independently changes each channel's distribution without preserving inter-channel relationships:

$$R' = T_R(R), \quad G' = T_G(G), \quad B' = T_B(B)$$

**Problem:** If the original image has $R \approx G$ (yellowish), but $T_R \neq T_G$, the equalized image may have $R' \neq G'$ — introducing false color shifts.

### Step 2: Luminance-Only Equalization

**Better approach:** Convert to a luminance-chrominance space (HSV, YCbCr, or LAB), equalize only the luminance channel, and convert back:

1. $I_{\text{RGB}} \to I_{\text{HSV}} = (H, S, V)$
2. $V' = T_V(V)$ (equalize value channel only)
3. $I_{\text{out}} = \text{HSV2RGB}(H, S, V')$

**Proof that this preserves hue and saturation:**
Only $V$ is modified; $H$ and $S$ are unchanged. Since hue is determined by the ratios of color channels (not their absolute values), and saturation measures deviation from gray (scaled by value), modifying $V$ while keeping $H, S$ constant preserves the color character. $\blacksquare$

### Step 3: The Histogram of the Luminance Channel

For natural images, the luminance histogram often approximates a log-normal or gamma distribution:

$$p_L(l) \approx \frac{1}{l\sigma\sqrt{2\pi}}\exp\left(-\frac{(\ln l - \mu)^2}{2\sigma^2}\right)$$

This skew toward darker values reflects the fact that most scenes contain more dark pixels than bright ones (shadows, backgrounds).

### Step 4: Multi-Histogram Equalization

For images with bimodal histograms (e.g., bright sky + dark ground), global equalization is suboptimal. **Bi-histogram equalization:**

1. Compute threshold $T = \text{mean}(I)$ (or use Otsu's method)
2. Split histogram at $T$: $h_L(k)$ for $k \leq T$, $h_U(k)$ for $k > T$
3. Equalize each sub-histogram independently
4. Map $[0, T]$ to $[0, T]$ and $[T+1, 255]$ to $[T+1, 255]$

**Advantage:** Preserves the mean brightness (unlike global HE which shifts the mean to $L/2$).

**Proof:** Each sub-histogram is equalized within its own range, so the fraction of pixels below $T$ remains the same: $\sum_{k=0}^T p'(k) = \sum_{k=0}^T p(k)$. Therefore the mean is approximately preserved. $\blacksquare$

### Step 5: RL Agent for Adaptive Equalization

An RL agent can select among equalization strategies:
$$\mathcal{A} = \{\text{global HE, per-channel HE, luminance HE, CLAHE, bi-HE, none}\}$$

The optimal choice depends on the image: high-contrast scenes benefit from CLAHE, low-contrast from global HE, and well-exposed images need no equalization. The agent learns this mapping from experience.

In [ ]:
# Create images with different histogram characteristics
size = 128

# Dark image (histogram clustered at low values)
dark_img = np.random.normal(50, 20, (size, size)).clip(0, 255).astype(np.uint8)

# Bright image (histogram clustered at high values)
bright_img = np.random.normal(200, 20, (size, size)).clip(0, 255).astype(np.uint8)

# Low contrast (narrow histogram)
low_contrast = np.random.normal(128, 10, (size, size)).clip(0, 255).astype(np.uint8)

# High contrast (wide histogram)
high_contrast = np.random.normal(128, 70, (size, size)).clip(0, 255).astype(np.uint8)

images = [dark_img, bright_img, low_contrast, high_contrast]
titles = ['Dark Image', 'Bright Image', 'Low Contrast', 'High Contrast']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, (img, title) in enumerate(zip(images, titles)):
    # Image
    axes[0, idx].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, idx].set_title(title)
    axes[0, idx].axis('off')
    
    # Histogram
    axes[1, idx].hist(img.ravel(), bins=256, range=(0, 255), 
                      color='steelblue', alpha=0.7, density=True)
    axes[1, idx].set_xlim(0, 255)
    axes[1, idx].set_xlabel('Pixel Intensity')
    axes[1, idx].set_ylabel('Probability')
    mu = img.mean()
    sigma = img.std()
    axes[1, idx].set_title(f'μ={mu:.0f}, σ={sigma:.0f}')
    axes[1, idx].axvline(mu, color='red', linestyle='--', label=f'μ={mu:.0f}')

plt.suptitle('Image Histograms — A Statistical Fingerprint of the Image',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('histograms.png', dpi=150, bbox_inches='tight')
plt.show()

## Histogram Backprojection — Object Detection via Color Distribution

Histogram backprojection uses the histogram of a known object to find that object in a new image. It is a simple but powerful technique for color-based object tracking.

### Step 1: Training Phase — Build Object Color Model

Given a region of interest (ROI) containing the target object, compute its normalized color histogram:

$$h_{\text{model}}(r, g) = \frac{|\{(x,y) \in \text{ROI} : R(x,y) = r, G(x,y) = g\}|}{|\text{ROI}|}$$

(Using 2D histogram in rg-chromaticity for illumination robustness)

### Step 2: Backprojection — Probability Map

For each pixel $(x, y)$ in the test image, assign it the probability that it belongs to the object:

$$B(x, y) = h_{\text{model}}(R(x,y), G(x,y))$$

**Result:** A probability map where bright pixels are likely to be part of the target object.

### Step 3: Proof of Optimality (Bayes' Rule)

**Theorem:** Histogram backprojection computes the posterior probability $P(\text{object} | \text{color})$.

**Proof:** By Bayes' rule:
$$P(\text{object} | c) = \frac{P(c | \text{object}) \cdot P(\text{object})}{P(c)}$$

If we assume uniform prior $P(\text{object})$ and approximate:
- $P(c | \text{object}) \propto h_{\text{model}}(c)$
- $P(c) \propto h_{\text{image}}(c)$

Then: $B(x,y) = \frac{h_{\text{model}}(c(x,y))}{h_{\text{image}}(c(x,y))}$ is proportional to the posterior. $\blacksquare$

The ratio histogram $h_{\text{model}} / h_{\text{image}}$ gives a better backprojection than using $h_{\text{model}}$ alone, because it accounts for the base rate of each color in the scene.

### Step 4: Mean-Shift Tracking with Backprojection

Combine backprojection with Mean-Shift for robust tracking:

1. Initialize search window at previous object location
2. Compute backprojection probability map
3. Find the centroid of the probability map within the window:
$$\bar{x} = \frac{\sum_{(x,y) \in W} x \cdot B(x,y)}{\sum_{(x,y) \in W} B(x,y)}, \quad \bar{y} = \frac{\sum_{(x,y) \in W} y \cdot B(x,y)}{\sum_{(x,y) \in W} B(x,y)}$$
4. Move window to $(\bar{x}, \bar{y})$
5. Repeat until convergence ($< \epsilon$ movement)

**Proof of convergence:** Mean-Shift is a gradient ascent on the kernel density estimate. Since the density is bounded, the sequence of centroids converges to a local mode. $\blacksquare$

### Step 5: RL Application

For an RL object-tracking agent:
- **State:** Backprojection map + current window position
- **Action:** Window shift $(\Delta x, \Delta y)$ and resize $(\Delta w, \Delta h)$
- **Reward:** IoU between tracking window and ground truth

The histogram backprojection provides the agent with a "likelihood map" — a much more informative state than raw pixels for tracking tasks.

In [ ]:
# Histogram Equalization — step by step
img = dark_img.copy()

# Step 1: Compute histogram
hist, bins = np.histogram(img.ravel(), bins=256, range=(0, 255))

# Step 2: Normalize to get PDF
pdf = hist / (size * size)

# Step 3: Compute CDF
cdf = np.cumsum(pdf)

# Step 4: Apply transform T(k) = round((L-1) * cdf(k))
T = np.round(255 * cdf).astype(np.uint8)

# Step 5: Apply to image
equalized = T[img]

# Visualize the process
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original
axes[0, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original (Dark)')
axes[0, 0].axis('off')

axes[0, 1].bar(range(256), pdf, color='steelblue', width=1)
axes[0, 1].set_title('Step 1-2: PDF p(k)')
axes[0, 1].set_xlabel('Intensity k')

axes[0, 2].plot(cdf, 'r-', linewidth=2)
axes[0, 2].set_title('Step 3: CDF c(k)')
axes[0, 2].set_xlabel('Intensity k')
axes[0, 2].set_ylabel('Cumulative probability')

axes[0, 3].plot(T, 'g-', linewidth=2)
axes[0, 3].plot([0, 255], [0, 255], 'k--', alpha=0.3, label='Identity')
axes[0, 3].set_title('Step 4: Transform T(k)')
axes[0, 3].set_xlabel('Input intensity')
axes[0, 3].set_ylabel('Output intensity')
axes[0, 3].legend()

# Equalized
axes[1, 0].imshow(equalized, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Step 5: Equalized!')
axes[1, 0].axis('off')

eq_hist, _ = np.histogram(equalized.ravel(), bins=256, range=(0, 255))
eq_pdf = eq_hist / (size * size)
axes[1, 1].bar(range(256), eq_pdf, color='green', width=1)
axes[1, 1].set_title('Equalized PDF (≈ Uniform)')

eq_cdf = np.cumsum(eq_pdf)
axes[1, 2].plot(eq_cdf, 'r-', linewidth=2)
axes[1, 2].plot([0, 255], [0, 1], 'k--', alpha=0.3, label='Ideal uniform CDF')
axes[1, 2].set_title('Equalized CDF (≈ Linear)')
axes[1, 2].legend()

axes[1, 3].text(0.5, 0.5, 
    'RL Insight:\n\nHistogram = Compact State\n(256 values vs M×N pixels)\n\n'
    'Equalization = One Action\n\n'
    'Reward = How uniform\nthe histogram becomes',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
    transform=axes[1, 3].transAxes)
axes[1, 3].axis('off')

plt.suptitle('Histogram Equalization — Complete Mathematical Pipeline',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('histogram_equalization.png', dpi=150, bbox_inches='tight')
plt.show()

## Mutual Information Between Color Channels

Mutual information measures the statistical dependence between image channels, providing a principled way to quantify redundancy in color representations.

### Step 1: Joint and Marginal Histograms

For a color image with channels $R$ and $G$, define:
- **Marginal PMF:** $p_R(r) = \frac{|\{(x,y) : R(x,y) = r\}|}{N}$
- **Joint PMF:** $p_{RG}(r, g) = \frac{|\{(x,y) : R(x,y) = r \text{ AND } G(x,y) = g\}|}{N}$

The joint histogram is a 2D array of size $L \times L$ (typically $256 \times 256$).

### Step 2: Mutual Information Definition and Derivation

$$I(R; G) = \sum_{r=0}^{L-1}\sum_{g=0}^{L-1} p_{RG}(r, g) \log_2 \frac{p_{RG}(r, g)}{p_R(r) \cdot p_G(g)}$$

**Equivalently, using entropies:**
$$I(R; G) = H(R) + H(G) - H(R, G)$$

**Proof of equivalence:**
$$I(R;G) = \sum_{r,g} p_{RG} \log \frac{p_{RG}}{p_R p_G} = \sum_{r,g} p_{RG} [\log p_{RG} - \log p_R - \log p_G]$$
$$= -H(R,G) - \sum_{r,g} p_{RG} \log p_R - \sum_{r,g} p_{RG} \log p_G$$
$$= -H(R,G) + H(R) + H(G) \quad \blacksquare$$

(using $\sum_g p_{RG}(r,g) = p_R(r)$ and similarly for $G$)

### Step 3: Properties of Mutual Information

1. **Non-negativity:** $I(R;G) \geq 0$ (by Gibbs' inequality / Jensen's inequality)
2. **Symmetry:** $I(R;G) = I(G;R)$
3. **Zero iff independent:** $I(R;G) = 0 \iff p_{RG}(r,g) = p_R(r) p_G(g)$ for all $r, g$
4. **Upper bound:** $I(R;G) \leq \min(H(R), H(G))$

**Proof of non-negativity (via Jensen's):**
$$I(R;G) = -\sum_{r,g} p_{RG} \log \frac{p_R p_G}{p_{RG}} \geq -\log\sum_{r,g} p_{RG} \cdot \frac{p_R p_G}{p_{RG}} = -\log 1 = 0 \quad \blacksquare$$

### Step 4: Interpretation for Image Analysis

- **High MI between R and G:** The channels are highly correlated (redundant) → PCA or decorrelation could compress the representation
- **Low MI:** Channels carry independent information → all channels needed

**Typical MI values for natural images:**
- $I(R;G) \approx 2.5$ bits (R and G are highly correlated in natural scenes)
- $I(R;B) \approx 1.5$ bits (less correlated)
- $I(G;B) \approx 1.8$ bits

### Step 5: MI as an Image Registration Metric

Mutual information is the gold standard for registering images from different modalities (e.g., MRI and CT scans) because it measures statistical dependence without assuming a linear relationship:

$$\hat{\mathbf{T}} = \arg\max_{\mathbf{T}} I(I_1; I_2 \circ \mathbf{T})$$

where $\mathbf{T}$ is the geometric transformation aligning the images. An RL agent can maximize MI as a reward signal for multi-modal image registration tasks.

## Otsu's Method — Optimal Threshold from Histogram Analysis

Otsu's method automatically determines the optimal threshold for binarizing a grayscale image by maximizing the between-class variance. It is a histogram-based algorithm that requires no parameters.

### Step 1: Two-Class Model

Given threshold $t$, pixels are divided into two classes:
- **Class 0** (background): pixels with intensity $\leq t$
- **Class 1** (foreground): pixels with intensity $> t$

**Class probabilities:**
$$\omega_0(t) = \sum_{k=0}^{t} p(k), \quad \omega_1(t) = \sum_{k=t+1}^{L-1} p(k) = 1 - \omega_0(t)$$

**Class means:**
$$\mu_0(t) = \frac{1}{\omega_0(t)}\sum_{k=0}^{t} k \cdot p(k), \quad \mu_1(t) = \frac{1}{\omega_1(t)}\sum_{k=t+1}^{L-1} k \cdot p(k)$$

### Step 2: Between-Class Variance

$$\sigma_B^2(t) = \omega_0(t) \omega_1(t) [\mu_0(t) - \mu_1(t)]^2$$

**Derivation:** The total variance decomposes as:
$$\sigma_T^2 = \sigma_W^2(t) + \sigma_B^2(t)$$

where $\sigma_W^2 = \omega_0 \sigma_0^2 + \omega_1 \sigma_1^2$ (within-class) and $\sigma_B^2 = \omega_0(\mu_0 - \mu_T)^2 + \omega_1(\mu_1 - \mu_T)^2$ (between-class).

Since $\sigma_T^2$ is constant (independent of $t$), maximizing $\sigma_B^2$ is equivalent to minimizing $\sigma_W^2$.

Using $\mu_T = \omega_0 \mu_0 + \omega_1 \mu_1$:
$$\sigma_B^2 = \omega_0(\mu_0 - \omega_0\mu_0 - \omega_1\mu_1)^2 + \omega_1(\mu_1 - \omega_0\mu_0 - \omega_1\mu_1)^2$$
$$= \omega_0 \omega_1^2(\mu_0 - \mu_1)^2 + \omega_1 \omega_0^2(\mu_1 - \mu_0)^2 = \omega_0\omega_1(\mu_0 - \mu_1)^2 \quad \blacksquare$$

### Step 3: Optimal Threshold

$$t^* = \arg\max_{0 \leq t < L} \sigma_B^2(t) = \arg\max_{t} \omega_0(t)\omega_1(t)[\mu_0(t) - \mu_1(t)]^2$$

**Efficient computation:** Both $\omega_0(t)$ and $\sum_{k=0}^t k \cdot p(k)$ can be updated incrementally as $t$ increases by 1, making the entire algorithm $O(L)$ — a single pass through the histogram.

### Step 4: Proof of Optimality (Fisher Discriminant)

Otsu's criterion is equivalent to the Fisher linear discriminant in 1D:

$$J(t) = \frac{\sigma_B^2(t)}{\sigma_W^2(t)}$$

**Theorem:** Maximizing $J(t)$ gives the threshold that best separates the two classes in the Fisher discriminant sense.

**Proof:** Since $\sigma_T^2 = \sigma_W^2 + \sigma_B^2$ is constant, $J(t) = \sigma_B^2/(\sigma_T^2 - \sigma_B^2)$ is monotonically increasing in $\sigma_B^2$. Thus maximizing $J(t)$ and $\sigma_B^2(t)$ give the same $t^*$. $\blacksquare$

### Step 5: RL Connection — Adaptive Thresholding

An RL agent for document binarization can use Otsu's threshold as a baseline action, then learn adjustments:
$$t_{\text{agent}} = t_{\text{Otsu}} + \Delta t$$

where $\Delta t$ is a learned correction. The agent surpasses Otsu by adapting to local image statistics, document type, and noise conditions.

## Histogram-Based Image Comparison — Statistical Distance Measures

Comparing images via their histograms is computationally efficient and provides rotation/translation invariant features. Here we derive the key histogram distance measures.

### Step 1: Histogram Intersection

$$d_{\text{int}}(h_1, h_2) = \sum_{k=0}^{L-1} \min(h_1(k), h_2(k))$$

**Interpretation:** The overlapping area between two histograms. Ranges from 0 (no overlap) to $\min(N_1, N_2)$ (identical).

**Normalized:** $\hat{d}_{\text{int}} = d_{\text{int}} / \min(\sum h_1, \sum h_2) \in [0, 1]$

### Step 2: Chi-Squared Distance

$$\chi^2(h_1, h_2) = \sum_{k=0}^{L-1} \frac{(h_1(k) - h_2(k))^2}{h_1(k) + h_2(k)}$$

**Derivation from the chi-squared test statistic:** Under the null hypothesis that both histograms come from the same distribution, $\chi^2$ follows the chi-squared distribution with $L-1$ degrees of freedom.

**Advantage over Euclidean:** The denominator normalizes by the bin size, so differences in rarely-used intensity levels are penalized more than differences in common levels.

### Step 3: Bhattacharyya Distance

$$d_B(h_1, h_2) = -\ln\left(\sum_{k=0}^{L-1} \sqrt{p_1(k) \cdot p_2(k)}\right)$$

where $p_i(k)$ are the normalized histograms.

**The Bhattacharyya coefficient** $BC = \sum_k \sqrt{p_1(k) p_2(k)}$ satisfies:

1. $0 \leq BC \leq 1$
2. $BC = 1$ iff $p_1 = p_2$ (identical)
3. $BC = 0$ iff $p_1$ and $p_2$ have disjoint support

**Proof of upper bound:** By Cauchy-Schwarz: $\sum \sqrt{p_1 p_2} \leq \sqrt{\sum p_1 \cdot \sum p_2} = 1$. $\blacksquare$

### Step 4: Kullback-Leibler Divergence

$$D_{KL}(p_1 \| p_2) = \sum_{k=0}^{L-1} p_1(k) \ln\frac{p_1(k)}{p_2(k)}$$

**Properties:**
- Non-negative: $D_{KL} \geq 0$ (Gibbs' inequality)
- NOT symmetric: $D_{KL}(p_1 \| p_2) \neq D_{KL}(p_2 \| p_1)$ in general
- Not a true metric (violates symmetry and triangle inequality)

**Symmetric variant (Jensen-Shannon divergence):**
$$D_{JS}(p_1, p_2) = \frac{1}{2}D_{KL}(p_1 \| m) + \frac{1}{2}D_{KL}(p_2 \| m), \quad m = \frac{p_1 + p_2}{2}$$

$D_{JS}$ IS a proper metric (when square-rooted).

### Step 5: RL Applications of Histogram Distances

| Task | Best Metric | Reason |
|:-----|:-----------|:-------|
| Image retrieval | Bhattacharyya | Robust to small shifts |
| Change detection | $\chi^2$ | Sensitive to distribution changes |
| Style transfer reward | JS divergence | Symmetric, bounded |
| Histogram matching | KL divergence | Information-theoretic optimality |

An RL agent performing content-based image retrieval can use histogram distances as compact state features:
$$\mathbf{s} = [d_B(h_{\text{query}}, h_i)]_{i=1}^{N} \in \mathbb{R}^N$$

This reduces image comparison to a vector of scalar distances — a much simpler state space than raw pixels.

## Local Histogram Processing — Adaptive Enhancement

Global histogram equalization applies a single transform to the entire image, which can fail when different regions have very different brightness distributions. Local (adaptive) methods use per-region histograms.

### Step 1: Sliding Window Histogram

For each pixel $(x, y)$, compute the histogram of a local window $W$ of size $(2r+1) \times (2r+1)$:

$$p_{x,y}(k) = \frac{|\{(i, j) \in W_{x,y} : I(i, j) = k\}|}{(2r+1)^2}$$

$$c_{x,y}(k) = \sum_{j=0}^{k} p_{x,y}(j)$$

Apply local equalization: $I_{\text{out}}(x, y) = (L-1) \cdot c_{x,y}(I(x, y))$

### Step 2: CLAHE — Contrast Limited Adaptive Histogram Equalization

Standard AHE can over-amplify noise in uniform regions. CLAHE clips the histogram to limit contrast amplification:

**Clip limit** $\beta$: any histogram bin with $p(k) > \beta / N_{\text{window}}$ is clipped. The excess counts are redistributed uniformly:

$$p_{\text{clipped}}(k) = \min(p(k), \beta)$$
$$\text{excess} = \sum_k \max(0, p(k) - \beta)$$
$$p_{\text{CLAHE}}(k) = p_{\text{clipped}}(k) + \frac{\text{excess}}{L}$$

**Proof that clipping limits contrast gain:**

The maximum slope of the equalization transform is $dT/dk = (L-1) \cdot p(k)$. Without clipping, $p(k)$ can be very large for peaked histograms, causing extreme contrast amplification. With clipping: $p_{\text{CLAHE}}(k) \leq \beta$, so $dT/dk \leq (L-1)\beta$, bounding the contrast gain. $\blacksquare$

### Step 3: Efficient Tile-Based Implementation

Instead of per-pixel computation ($O(N^2 r^2)$ — too slow):

1. Divide image into non-overlapping tiles (e.g., $8 \times 8$ tiles)
2. Compute CLAHE transform for each tile: $T_{\text{tile}}(k)$
3. For pixels between tiles, use **bilinear interpolation** of the four nearest tile transforms:

$$T(k; x, y) = (1-\alpha)(1-\beta)T_1(k) + \alpha(1-\beta)T_2(k) + (1-\alpha)\beta T_3(k) + \alpha\beta T_4(k)$$

where $\alpha, \beta \in [0, 1]$ are the fractional positions between tile centers.

**Complexity:** $O(N^2 + T \cdot L)$ where $T$ is the number of tiles — essentially linear in image size.

### Step 4: CLAHE Parameters as RL Actions

An RL agent for image enhancement can treat CLAHE parameters as continuous actions:

$$\mathcal{A} = \{(\text{clip limit } \beta, \text{ tile size } t, \text{ contrast limit } c)\}$$

The agent learns to set these parameters adaptively for each image, outperforming any fixed parameter choice. The reward is a combination of perceived quality metrics:
$$R = \alpha \cdot \text{contrast\_improvement} + \beta \cdot \text{naturalness} - \gamma \cdot \text{noise\_amplification}$$

---
**Module 1 Complete!** Next → Module 2: Image Processing Basics